In [ ]:
import os, sys
import logging
import k3d
from functools import partial
import open3d as o3d

import torch 
import numpy as np 
import matplotlib.colors as mcolors

from create_example_object import *

# Import kaolin physics
import kaolin.physics as physics
print(dir(physics.simplicits.network))
# Imports for displaying the object
from scipy.spatial import Delaunay
import ipywidgets as widgets
from IPython.display import display

device = 'cuda'
dtype = torch.float32

logging.basicConfig(level=logging.INFO, stream=sys.stdout)
logger = logging.getLogger(__name__)

# Run training from here

In [ ]:
NUM_HANDLES = 5
NUM_STEPS = 10000
LR_START = 1e-3
NUM_SAMPLES = 1000

ENERGY_INTERP_LINSPACE = np.linspace(0, 1, NUM_STEPS, endpoint=False)
from physics.loss import compute_losses_H

so_mesh, so_pts, so_yms, so_prs, so_rhos, so_appx_vol = example_unit_cube_object_mesh(resolution=15)
so_model = physics.simplicits.network.SimplicitsMLP(spatial_dimensions=3, layer_width=64, num_handles=NUM_HANDLES, num_layers=8)
so_optimizer = torch.optim.Adam(so_model.parameters(), LR_START)

# Don't normalize object to unit cube unless is very small or very lage
# so_bb_pts = (torch.min(so_pts, axis=0).values, torch.max(so_pts, axis=0).values)
# so_normalized_pts = (so_pts - so_bb_pts[0])/(so_bb_pts[1] - so_bb_pts[0])

so_pts = torch.as_tensor(so_pts, device=device, dtype=dtype)
so_yms = torch.as_tensor(so_yms, device=device, dtype=dtype).unsqueeze(-1)
so_prs = torch.as_tensor(so_prs, device=device, dtype=dtype).unsqueeze(-1)
so_rhos = torch.as_tensor(so_rhos, device=device, dtype=dtype).unsqueeze(-1)
so_normalized_pts = so_pts.clone()
so_model.to(device)

partial_compute_losses = partial(compute_losses_H, 
                             batch_size=10, 
                             num_handles=NUM_HANDLES, 
                             appx_vol=1, 
                             num_samples=NUM_SAMPLES, 
                             le_coeff=1e-1, 
                             lo_coeff=1e6)

so_model.train()
for i in range(NUM_STEPS):
    #Set grads to zero
    so_optimizer.zero_grad()
    #train a step
    lh, lo = partial_compute_losses(so_model, so_normalized_pts, so_yms, so_prs, so_rhos, float(i/NUM_STEPS))
    loss = lh + lo
    # Backprop over the losses
    loss.backward()
    # Take optimizer step
    so_optimizer.step()
    
    if i%100 == 0:
        print(f'Training step: {i}, lh: {le.item()}, lo: {lo.item()}')

so_model.eval()

In [ ]:
# visualize lbs weights

sample_indices = torch.randint(low=0, high=so_normalized_pts.shape[0], size=(NUM_SAMPLES,), device=so_normalized_pts.device)
sample_pts = so_normalized_pts[sample_indices]
sample_yms = so_yms[sample_indices]
sample_prs = so_prs[sample_indices]
sample_rhos = so_rhos[sample_indices]

weight = so_model(sample_pts)

batch_transforms = 0.1 * torch.randn(10, NUM_HANDLES, 3, 4, dtype=so_normalized_pts.dtype).to(so_normalized_pts.device)

print(sample_pts.shape)
print(weight.shape)
print(batch_transforms.shape)
print(so_rhos.shape)

def scalar_to_rgb(s_normalized, mincolorstr, maxcolorstr, minval=None, maxval=None):
    cmap = mcolors.LinearSegmentedColormap.from_list("", [mincolorstr, maxcolorstr])

    if(np.sum(s_normalized)==0):
        return cmap(s_normalized)[:, 0:3]*255
    else:
        if minval == None:
            minval = np.min(s_normalized)
        if maxval == None:
            maxval = np.max(s_normalized)
        s_normalized = (s_normalized - np.min(s_normalized)) / (maxval - minval)
        colors_rgb = cmap(s_normalized)[:, 0:3]*255
        return colors_rgb

def rgb2hex(rgb):
    str_res = "0x{0:02x}{1:02x}{2:02x}".format(int(rgb[0]), int(rgb[1]), int(rgb[2]))
    return int(str_res, 16)

modes = so_model(so_pts)

def visualize_mode(viz_mode, modes):
    rgb = scalar_to_rgb(modes[:,viz_mode].detach().cpu().numpy(), "blue", "red")
    colors = [rgb2hex(rgb[xx, :]) for xx in range(rgb.shape[0])]
    return colors

# Viewing 3 of the weight functions
# Plot more to view more
plot = k3d.plot()
plot += k3d.points(so_normalized_pts.detach().cpu().numpy(), colors=visualize_mode(0,modes),  point_size=0.01)
plot += k3d.points(so_normalized_pts.detach().cpu().numpy(), colors=visualize_mode(4,modes),  point_size=0.01)
plot += k3d.points(so_normalized_pts.detach().cpu().numpy(), colors=visualize_mode(2,modes),  point_size=0.01)
plot.display()


# Run simulation from here

In [ ]:
# Sim parameters
NUM_STEPS = 100

FLOOR_PLANE = -1
PENALTY_WEIGHT = 10000
NUM_SAMPLES = 1000
device = 'cuda'
dtype = torch.float32
MAX_NEWTON_STEPS=5
MAX_LS_STEPS = 30

# Select the cubature points and corresponding material parameters
sample_indices = torch.randint(low=0, high=so_normalized_pts.shape[0], size=(NUM_SAMPLES,), device=so_normalized_pts.device)
sim_pts = so_pts[sample_indices]
sim_normalized_pts = so_normalized_pts[sample_indices]
sim_yms = 0.1*so_yms[sample_indices]
sim_prs = so_prs[sample_indices]
sim_rhos = 5*so_rhos[sample_indices]
sim_weights = torch.cat((so_model(sim_normalized_pts), torch.ones((sim_normalized_pts.shape[0], 1), device=device)), dim=1)
model_plus_rigid = lambda pts: torch.cat((so_model(pts), torch.ones((pts.shape[0], 1), device=device)), dim=1)

# Initialize simulation DOFs
z = torch.zeros(sim_weights.shape[1]*12 , dtype=dtype, device = device).unsqueeze(-1)
z_prev = z.clone().detach()
z_dot = torch.zeros_like(z, device=device)
x0_flat = sim_pts.flatten().unsqueeze(-1)


M, invM = physics.simplicits.precomputed.lumped_mass_matrix(sim_rhos, so_appx_vol, dim = 3)
dFdz = physics.simplicits.precomputed.jacobian_dF_dz(model_plus_rigid, sim_normalized_pts, z).detach()
dxdz = torch.autograd.functional.jacobian(lambda x: physics.simplicits.utils.weight_function_lbs(sim_pts, tfms = x.reshape(-1,3,4).unsqueeze(0), fcn = model_plus_rigid).flatten(), z.flatten())
bigI = torch.tile(torch.eye(3, device=device).flatten().unsqueeze(dim=1), (NUM_SAMPLES,1)).detach()
B = physics.simplicits.precomputed.lbs_matrix(sim_pts, sim_weights).detach()

# 3*num samples gravities per sample point
grav = torch.tensor([0, 9.8, 0], device=device)

BMB = B.T @ M @ B
BinvMB = B.T @ invM @ B

print(" Density: ",str(sim_rhos[0].item())+"kg/m^3\n", 
      "Youngs Mod: ", str(sim_yms[0].item())+"Pa\n", 
      "Poiss Ratio: ", str(sim_prs[0].item())+"\n", 
      "Appx Vol: ", str(so_appx_vol)+"m^3\n")

#########################SETUP MATERIAL AND SCENE FORCES######################################################
mus, lams = physics.materials.utils.to_lame(sim_yms, sim_prs)
material_object = physics.materials.NeohookeanMaterial(sim_yms, sim_prs)
gravity_object = physics.utils.Gravity(rhos=sim_rhos, acceleration=grav)
floor_object = physics.utils.Floor(floor_height=FLOOR_PLANE, floor_axis=1)
bdry_cond = physics.utils.Boundary()
bdry_indx = torch.nonzero(sim_pts[:,1]>1.45, as_tuple=False).squeeze()
bdry_pos = sim_pts[bdry_indx,:]
bdry_cond.set_pinned_verts(bdry_indx, bdry_pos)
integration_sampling = torch.as_tensor(so_appx_vol/NUM_SAMPLES, device=device, dtype=sim_pts.dtype)

#######################Physics Energy, Forces, Hessians########################################################
partial_bdry_e = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_energy(bdry_cond, B, coeff=PENALTY_WEIGHT, integration_sampling=None)
partial_bdry_g = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_gradient(bdry_cond, B, coeff=PENALTY_WEIGHT, integration_sampling=None)
partial_bdry_h = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_hessian(bdry_cond, B, coeff=PENALTY_WEIGHT, integration_sampling=None)

partial_grav_e = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_energy(gravity_object, B, coeff=1, integration_sampling=integration_sampling) 
partial_grav_g = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_gradient(gravity_object, B, coeff=1, integration_sampling=integration_sampling) 
partial_grav_h = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_hessian(gravity_object, B, coeff=1, integration_sampling=integration_sampling) 

partial_material_e = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_material_energy(material_object, dFdz, coeff=1, integration_sampling=integration_sampling)
partial_material_g = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_material_gradient(material_object, dFdz, coeff=1, integration_sampling=integration_sampling)
partial_material_h = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_material_hessian(material_object, dFdz, coeff=1, integration_sampling=integration_sampling)

partial_floor_e = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_energy(floor_object, B, coeff=PENALTY_WEIGHT, integration_sampling=None)
partial_floor_g = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_gradient(floor_object, B, coeff=PENALTY_WEIGHT, integration_sampling=None)
partial_floor_h = physics.simplicits.simplicits_scene_forces.generate_fcn_simplicits_scene_hessian(floor_object, B, coeff=PENALTY_WEIGHT, integration_sampling=None)
###############################################################################

####################Backwards Euler Functions###########################################################
def potential_sum(output, z, z_dot, B, dFdz, x0_flat, bigI, defo_grad_fcns = [], pt_wise_fcns = []):
    # updates the quantity calculated in the output value
    F_ele = torch.matmul(dFdz, z) + bigI
    x_flat = B @ z + x0_flat
    x = x_flat.reshape(-1,3)
    for e in defo_grad_fcns:
        output += e(F_ele)
    for e in pt_wise_fcns:
        output += e(x)
        
def newton_E(z, z_prev, z_dot, B, BMB, dt, x0_flat, dFdz, bigI, defo_grad_energies = [], pt_wise_energies = []):
    pe_sum = torch.tensor([0], device=device, dtype=dtype)
    potential_sum(pe_sum, z, z_dot, B, dFdz, x0_flat, bigI, defo_grad_energies, pt_wise_energies)
    return 0.5 * z.T @ BMB @ z - z.T @ BMB @ z_prev - dt * z.T @ BMB @ z_dot + dt * dt * pe_sum

def newton_G(z, z_prev, z_dot, B, BMB, dt, x0_flat, dFdz, bigI, defo_grad_gradients = [], pt_wise_gradients = []):
    pe_grad_sum = torch.zeros_like(z)
    potential_sum(pe_grad_sum, z, z_dot, B, dFdz, x0_flat, bigI, defo_grad_gradients, pt_wise_gradients)
    return BMB @ z - BMB @ z_prev - dt * BMB @ z_dot + dt * dt * pe_grad_sum

def newton_H(z, z_prev, z_dot, B, BMB, dt, x0_flat, dFdz, bigI, defo_grad_hessians = [], pt_wise_hessians = []):
    pe_hess_sum = torch.zeros(z.shape[0], z.shape[0], device=device, dtype=dtype)
    potential_sum(pe_hess_sum, z, z_dot, B, dFdz, x0_flat, bigI, defo_grad_hessians, pt_wise_hessians)
    return BMB  + dt * dt * pe_hess_sum
    
##########################Backwards Euler Partials#####################################################
partial_newton_E = partial(newton_E, 
                           B=B.detach(), 
                           BMB = BMB.detach(), 
                           dt=DT, 
                           x0_flat=x0_flat.detach(), 
                           dFdz=dFdz.detach(), 
                           bigI=bigI.detach(),
                           defo_grad_energies=[partial_material_e],
                           pt_wise_energies=[partial_grav_e, partial_floor_e, partial_bdry_e])
partial_newton_G = partial(newton_G, 
                           B=B.detach(), 
                           BMB = BMB.detach(), 
                           dt=DT, 
                           x0_flat=x0_flat.detach(), 
                           dFdz=dFdz.detach(), 
                           bigI=bigI.detach(),
                           defo_grad_gradients=[partial_material_g],
                           pt_wise_gradients=[partial_grav_g, partial_floor_g, partial_bdry_g])
partial_newton_H = partial(newton_H, 
                           B=B.detach(), 
                           BMB = BMB.detach(), 
                           dt=DT, 
                           x0_flat=x0_flat.detach(), 
                           dFdz=dFdz.detach(), 
                           bigI=bigI.detach(),
                           defo_grad_hessians=[partial_material_h],
                           pt_wise_hessians=[partial_grav_h, partial_floor_h, partial_bdry_h])


# Displaying The Simulation 
## Using Hamilton + Symplectic Euler
Next we use K3D to display the simulation.
Scrub through the `TIMESTEP` slider below to see each step.

In [ ]:
# use the Symplectic Euler method:
# z: DOFs; p: momentum
# H = T(p) + V(z); T(p) = 0.5 * p^T M^-1 p, kinetic energy; V(z) = elastic energy + gravity + floor + boundary

z = torch.zeros(sim_weights.shape[1]*12 , dtype=dtype, device = device).unsqueeze(-1)


def compute_grad_potential(z, B, x0_flat, dFdz, bigI,
                           partial_material_e,
                           partial_grav_e,
                           partial_floor_e,
                           partial_bdry_e
                           ):
    """
    Compute ∇\Pi(z), total potential gradient wrt z.

    Args:
        z (torch.Tensor): DOF vector, shape (3*4*12, 1)
        B (torch.Tensor): Matrix that encodes the lbs transformation, given a set of vertices and corresponding weights, shape :math:`(3*\text{num_samples}, 12*\text{num_handles})`
        
    Returns:
        grad (torch.Tensor): Gradient of \Pi(z)

    """
    z = z.detach().clone().requires_grad_(True)

    pe_sum = torch.tensor([0], device=device, dtype=dtype)
    potential_sum(pe_sum, z, z_dot=None, B=B, dFdz=dFdz, x0_flat=x0_flat, bigI=bigI, 
        defo_grad_fcns=[partial_material_e], 
        pt_wise_fcns=[partial_grav_e, partial_floor_e, partial_bdry_e]
        )


    # compute gradient
    grad = torch.autograd.grad(pe_sum, z)[0]

    return grad

compute_grad_potential_fn = partial(
    compute_grad_potential,
    B=B.detach(),
    x0_flat=x0_flat.detach(),
    dFdz=dFdz.detach(),
    bigI=bigI.detach(),
    partial_material_e=partial_material_e,
    partial_grav_e=partial_grav_e,
    partial_floor_e=partial_floor_e,
    partial_bdry_e=partial_bdry_e
)

In [ ]:
z = torch.zeros(sim_weights.shape[1]*12 , dtype=dtype, device = device).unsqueeze(-1)
p = torch.zeros_like(z, device=device)

# set each time step
DT = 0.005
NUM_STEPS = 10000

# explicit and controllable energy dissipation, support more complex energy transform function
damping_factor = 0.9995
# damping_factor = 0.9998


states = [z.clone().detach()]
# p = -DT * compute_grad_potential_fn(z)

total_energy = []

for time_step in range(int(NUM_STEPS)):

    # update Hailtonian
    grad_V = compute_grad_potential_fn(z)

    p = p - DT * grad_V
    p = damping_factor * p  # apply damping
    z = z + DT * torch.linalg.solve(BinvMB, p)

    # kinetic energy
    # T(p) = 0.5 * p^T BinvMB p
    kinetic_energy = 0.5 * p.T @ torch.linalg.solve(BinvMB, p)

    F_ele = torch.matmul(dFdz, z) + bigI
    x_pts = (B @ z + x0_flat).reshape(-1,3)

    print(f'\tFloor E:{partial_floor_e(x_pts).item()}, Grav E:{partial_grav_e(x_pts).item()}, Bdry E:{partial_bdry_e(x_pts).item()}, Elastic E:{partial_material_e(F_ele).item()}, Kinetic E:{kinetic_energy.item()}')

    # total_energy = [systemtotal_energy, potential_energy, total energy inside body, elastic_energy, kinetic_energy]
    # elastic_energy = [material_energy]
    total_energy.append([
        partial_floor_e(x_pts).item() + partial_grav_e(x_pts).item() + partial_bdry_e(x_pts).item() + partial_material_e(F_ele).item() + kinetic_energy.item(),
        partial_floor_e(x_pts).item() + partial_grav_e(x_pts).item() + partial_bdry_e(x_pts).item(), 
        partial_material_e(F_ele).item() + kinetic_energy.item(), 
        partial_material_e(F_ele).item(),
        kinetic_energy.item()])

    states.append(z.clone().detach())

total_energy = np.array(total_energy)

# Plot the energy 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(total_energy[:,3], c='blue')

plt.text(0.5, 0.97, 'Elastic energy transform over time.', 
         transform=plt.gca().transAxes, fontsize=16, 
         ha='center', va='top', weight='bold')
plt.text(0.53, 0.02, 'Time step', 
         transform=plt.gca().transAxes, fontsize=14, 
         ha='right', va='bottom', weight='bold')
plt.text(0.01, 0.55, 'Energy', 
         transform=plt.gca().transAxes, fontsize=14, 
         ha='left', va='top', weight='bold', rotation=90)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

total_energy_array = np.array(total_energy)

start = 6960
end = 8100

kinetic_energy = total_energy_array[:,3]
kinetic_energy = kinetic_energy[start:end]
elastic_energy = np.array(total_energy[:,3])
elastic_energy = elastic_energy[start:end]

tick_position = np.linspace(0, kinetic_energy.shape[0], 5)
tick_labels = np.linspace(start, end, 5).astype(int)

insidebody_energy = elastic_energy + kinetic_energy

plt.figure(figsize=(12, 8))


plt.subplot(2, 1, 1)
# plt.plot(elastic_energy, 'b-', linewidth=2)
# plt.plot(kinetic_energy, 'g-', linewidth=2)
plt.plot(insidebody_energy, 'b-', linewidth=2)
plt.xticks(tick_position, tick_labels)

plt.text(0.15, 0.02, 'Time step', 
         transform=plt.gca().transAxes, fontsize=14, 
         ha='right', va='bottom', weight='bold')
plt.ylabel('Energy', fontsize=14, weight='bold')
plt.title('Time Evolution of Internal Energy with Dissipation Envelope', fontsize=16, weight='bold')
plt.grid(True, alpha=0.3)

# Envelope of oscillation
from scipy.signal import hilbert
analytic_signal = hilbert(insidebody_energy - np.mean(insidebody_energy))
envelope = np.abs(analytic_signal) + np.mean(insidebody_energy)
plt.plot(envelope, 'r--', linewidth=2, alpha=0.7, label='Decay Envelope')
plt.plot(-envelope + 2*np.mean(insidebody_energy), 'r--', linewidth=2, alpha=0.7)
plt.legend(loc='upper right')

plt.subplot(2, 1, 2)
# Phase space: Energy vs Energy Rate
energy_rate = np.gradient(insidebody_energy)
plt.plot(insidebody_energy, energy_rate, 'b-', alpha=0.7, linewidth=1)
plt.text(0.15, 0.02, 'Energy', 
         transform=plt.gca().transAxes, fontsize=14, 
         ha='right', va='bottom', weight='bold')
plt.ylabel('dE/dt', fontsize=14, weight='bold')
plt.title('Phase Space', fontsize=16, weight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualization of particle view for each integration

In [ ]:
def generate_ground_plane(plane):
    # Define the size of the ground plane
    x_size = 1
    y_size = 1
    num_points_x = 30
    num_points_y = 30
    
    # Generate grid points
    x = np.linspace(-x_size/2, x_size/2, num_points_x)
    y = np.linspace(-y_size/2, y_size/2, num_points_y)
    X, Y = np.meshgrid(x, y)
    
    # Generate vertices
    vertices = np.column_stack([X.flatten(), plane * np.ones_like(X.flatten()), Y.flatten()])
    vertices += +1e-3*np.random.rand(vertices.shape[0], vertices.shape[1])
    # Perform Delaunay triangulation
    tri = Delaunay(vertices)
    return torch.tensor(vertices), torch.tensor(tri.simplices)

v, f = generate_ground_plane(FLOOR_PLANE)
def reconstruct_deformed_positions(sample_pts, weights, batch_transforms):
    # sample_pts: (N, 3)
    # weights: (N, H)
    # batch_transforms: (B, H, 3, 4)  # assuming B = 1 or broadcast
    
    N, H = weights.shape
    B = batch_transforms.shape[0]
    device = sample_pts.device

    # Add 1 to each point to make it homogeneous (N, 4)
    sample_pts_h = torch.cat([sample_pts, torch.ones_like(sample_pts[..., :1])], dim=-1)  # (N, 4)

    # Expand dims to align for broadcasting
    pts_exp = sample_pts_h.unsqueeze(1).expand(-1, H, -1)        # (N, H, 4)
    trans_exp = batch_transforms[0] if B == 1 else batch_transforms  # (H, 3, 4)
    
    # Apply transform: (N, H, 3) = sum over i: w_i * (T_i @ x)
    transformed = torch.matmul(trans_exp, pts_exp.transpose(-1, -2)).transpose(-1, -2)  # (N, H, 3)
    
    # Apply LBS weights
    x_deformed = torch.sum(weights.unsqueeze(-1) * transformed, dim=1)  # (N, 3)
    
    return x_deformed
# Function to create points
def create_points(t):
    z = states[int(t)]
    x = B @ z + x0_flat
    # print(torch.norm(x - x0_flat))
    scene_verts = x.reshape(-1,3).unsqueeze(0).cpu().detach()
    return scene_verts.cpu().detach().numpy()

# Create a plot
plot = k3d.plot(camera_auto_fit=False)
floor_plot = k3d.points(v.cpu().detach().numpy(), point_size=0.02, color=0x00ff00, shader='3d')
plot += floor_plot
# Initial set of points
initial_points = create_points(0)
points_plot = k3d.points(initial_points, point_size=0.03, color=0x0000ff, shader='3d')
plot += points_plot

# Rotate the camera so that y is the vertical axis
plot.camera = [1.5, 1, 1.5, 0, -0.25, 0, 0, 1, 0]
plot.grid_visible = False
plot.axes_helper = False
plot.menu_visibility = False
# plot.camera = [3, 1, 0, 0, 0, 0, 0, 1, 0]  # (eye_x, eye_y, eye_z, target_x, target_y, target_z, up_x, up_y, up_z)


# Display the plot
plot.display()

# Define the function to update the points
def update_points(TIMESTEP):
    new_points = create_points(TIMESTEP)
    points_plot.positions = new_points.astype(np.float32)

# Create a slider
slider = widgets.FloatSlider(min=0, max=len(states), step=1, value=0)

# Link the slider to the update function
widgets.interactive(update_points, TIMESTEP=slider)

# Display the slider
display(slider)

In [ ]:
# for fectching k3d screenshot

import ipywidgets
out = ipywidgets.Output()
from scipy.spatial import Delaunay
import k3d
from kaolin.physics.simplicits.utils import weight_function_lbs

# vertices solver
def create_points(t):
    z = states[int(t)]
    x = B @ z + x0_flat
    # print(torch.norm(x - x0_flat))
    scene_verts = x.reshape(-1,3).unsqueeze(0).cpu().detach()
    return scene_verts.cpu().detach().numpy()

# selected timestep
screenshot_timestep = [0, 161, 242, 282, 363, 524, 806, 887, 2016, 2943, 3870, 5000]

# generate floor
def generate_ground_plane(plane):
    # Define the size of the ground plane
    x_size = 1
    y_size = 1
    num_points_x = 30
    num_points_y = 30
    
    # Generate grid points
    x = np.linspace(-x_size/2, x_size/2, num_points_x)
    y = np.linspace(-y_size/2, y_size/2, num_points_y)
    X, Y = np.meshgrid(x, y)
    
    # Generate vertices
    vertices = np.column_stack([X.flatten(), plane * np.ones_like(X.flatten()), Y.flatten()])
    vertices += +1e-3*np.random.rand(vertices.shape[0], vertices.shape[1])
    # Perform Delaunay triangulation
    tri = Delaunay(vertices)
    return torch.tensor(vertices), torch.tensor(tri.simplices)

# floor vertices, faces
v, f = generate_ground_plane(-1)

# reset variable
screenshot_plot = None

# first, create a screenshot plot with floor mesh
screenshot_plot = k3d.plot(camera_auto_fit=False)
screenshot_plot.camera = [1.5, 1, 1.5, 0, -0.25, 0, 0, 1, 0]
screenshot_plot.grid_visible = False
screenshot_plot.axes_helper = False
screenshot_plot.menu_visibility = False

# Create floor particle plot
floorpoints_plot = k3d.points(v.cpu().detach().numpy(), point_size=0.02, color=0x00ff00, shader='3d')
screenshot_plot += floorpoints_plot

# Initial set of points
newpoints = create_points(0)
objpoints_plot = k3d.points(newpoints.astype(np.float32), point_size=0.03, color=0x0000ff, shader='3d')
screenshot_plot += objpoints_plot

screenshot_plot.display()

# run in background and get screenshot
@screenshot_plot.yield_screenshots
def coroutine():
    for t in screenshot_timestep:
        
        # solve vertices position and update
        newpoints = create_points(t)
        objpoints_plot.positions = newpoints.astype(np.float32)

        # create screenshot
        screenshot_plot.fetch_screenshot()
        screenshot = yield

        with open(f'./tmp/screenshot{t}.png', 'wb') as f:
            f.write(screenshot)
        with out:
            print(f'./tmp/screenshot{t}.png saved')

    with out:
        print('done')

coroutine()
print('runing in background')
out

# Visualization of mesh view for each integration

In [ ]:
from kaolin.physics.simplicits.utils import weight_function_lbs
def get_object_deformed_pts(points, z):

    return weight_function_lbs(points,
                               tfms=z.reshape(-1, 3, 4).unsqueeze(0), 
                               fcn=lambda pts: torch.cat((so_model(pts), torch.ones((pts.shape[0], 1), device='cuda')), dim=1)
                               )

z_0 = states[3900]
mesh_v = so_mesh.vertices.clone()

mesh_v_i = get_object_deformed_pts(mesh_v, z_0)
mesh_v_i = mesh_v_i.squeeze(1).squeeze(1).cpu().detach().numpy()

print(mesh_v)
print(mesh_v_i)

plot = k3d.plot(camera_auto_fit=False)
mesh_plot = k3d.mesh(mesh_v_i, so_mesh.faces.cpu().detach().numpy(), color=0x00ff00, shader='3d')
plot += mesh_plot
plot.display()

In [ ]:
original_vertices = so_mesh.vertices.clone()

mesh_v = so_mesh.vertices.clone()
mesh_f = so_mesh.faces.clone()

from scipy.spatial import Delaunay
import k3d
import ipywidgets as widgets
from kaolin.physics.simplicits.utils import weight_function_lbs

def get_object_deformed_pts(points, z):

    return weight_function_lbs(points,
                               tfms=z.reshape(-1, 3, 4).unsqueeze(0), 
                               fcn=lambda pts: torch.cat((so_model(pts), torch.ones((pts.shape[0], 1), device='cuda')), dim=1)
                               )

def generate_ground_plane(plane):
    # Define the size of the ground plane
    x_size = 1
    y_size = 1
    num_points_x = 30
    num_points_y = 30
    
    # Generate grid points
    x = np.linspace(-x_size/2, x_size/2, num_points_x)
    y = np.linspace(-y_size/2, y_size/2, num_points_y)
    X, Y = np.meshgrid(x, y)
    
    # Generate vertices
    vertices = np.column_stack([X.flatten(), plane * np.ones_like(X.flatten()), Y.flatten()])
    vertices += +1e-3*np.random.rand(vertices.shape[0], vertices.shape[1])
    # Perform Delaunay triangulation
    tri = Delaunay(vertices)
    return torch.tensor(vertices), torch.tensor(tri.simplices)

v, f = generate_ground_plane(-1)
def reconstruct_deformed_positions(sample_pts, weights, batch_transforms):
    # sample_pts: (N, 3)
    # weights: (N, H)
    # batch_transforms: (B, H, 3, 4)  # assuming B = 1 or broadcast
    
    N, H = weights.shape
    B = batch_transforms.shape[0]
    device = sample_pts.device

    # Add 1 to each point to make it homogeneous (N, 4)
    sample_pts_h = torch.cat([sample_pts, torch.ones_like(sample_pts[..., :1])], dim=-1)  # (N, 4)

    # Expand dims to align for broadcasting
    pts_exp = sample_pts_h.unsqueeze(1).expand(-1, H, -1)        # (N, H, 4)
    trans_exp = batch_transforms[0] if B == 1 else batch_transforms  # (H, 3, 4)
    
    # Apply transform: (N, H, 3) = sum over i: w_i * (T_i @ x)
    transformed = torch.matmul(trans_exp, pts_exp.transpose(-1, -2)).transpose(-1, -2)  # (N, H, 3)
    
    # Apply LBS weights
    x_deformed = torch.sum(weights.unsqueeze(-1) * transformed, dim=1)  # (N, 3)
    
    return x_deformed
# Function to create points
def create_points(t):
    z = states[int(t)]
    # x = B @ z + x0_flat
    # print(torch.norm(x - x0_flat))

    mesh_v_i = get_object_deformed_pts(mesh_v, z)
    mesh_v_i = mesh_v_i.squeeze(1).squeeze(1).cpu().detach().numpy()

    return mesh_v_i

# Create a plot
mesh_plot = k3d.plot(camera_auto_fit=False)
floormesh_plot = k3d.mesh(v.cpu().detach().numpy(), f.cpu().detach().numpy(), color=0x0000ff, shader='3d')
mesh_plot += floormesh_plot
# Initial set of points
initial_points = create_points(0)
objmesh_plot = k3d.mesh(initial_points, so_mesh.faces.cpu().detach().numpy(), color=0x00ff00, shader='3d')
mesh_plot += objmesh_plot

# Rotate the camera so that y is the vertical axis
mesh_plot.camera = [1.5, 1, 1.5, 0, -0.25, 0, 0, 1, 0]
mesh_plot.grid_visible = False
mesh_plot.axes_helper = False
mesh_plot.menu_visibility = False
# mesh_plot.camera = [3, 1, 0, 0, 0, 0, 0, 1, 0]  # (eye_x, eye_y, eye_z, target_x, target_y, target_z, up_x, up_y, up_z)

# Display the plot
mesh_plot.display()

def update_mesh_plot(TIMESTEP):

    new_vertices = create_points(TIMESTEP)
    objmesh_plot.vertices = new_vertices.astype(np.float32)

# Define the function to update the points

# Create a slider
# slider = widgets.FloatSlider(min=0, max=len(states), step=1, value=0)
slider = widgets.FloatSlider(min=0, max=len(states), step=1, value=0)

# Link the slider to the update function
widgets.interactive(update_mesh_plot, TIMESTEP=slider)

# Display the slider
display(slider)

In [ ]:
import ipywidgets
out = ipywidgets.Output()
from scipy.spatial import Delaunay
import k3d
import ipywidgets as widgets
from kaolin.physics.simplicits.utils import weight_function_lbs

# vertices solver
def get_object_deformed_pts(points, z):

    return weight_function_lbs(points,
                               tfms=z.reshape(-1, 3, 4).unsqueeze(0), 
                               fcn=lambda pts: torch.cat((so_model(pts), torch.ones((pts.shape[0], 1), device='cuda')), dim=1)
                               )

# selected timestep
screenshot_timestep = [0, 161, 242, 282, 363, 524, 806, 887, 2016, 2943, 3870, 5000]

# generate floor
def generate_ground_plane(plane):
    # Define the size of the ground plane
    x_size = 1
    y_size = 1
    num_points_x = 30
    num_points_y = 30
    
    # Generate grid points
    x = np.linspace(-x_size/2, x_size/2, num_points_x)
    y = np.linspace(-y_size/2, y_size/2, num_points_y)
    X, Y = np.meshgrid(x, y)
    
    # Generate vertices
    vertices = np.column_stack([X.flatten(), plane * np.ones_like(X.flatten()), Y.flatten()])
    vertices += +1e-3*np.random.rand(vertices.shape[0], vertices.shape[1])
    # Perform Delaunay triangulation
    tri = Delaunay(vertices)
    return torch.tensor(vertices), torch.tensor(tri.simplices)

# floor vertices, faces
v, f = generate_ground_plane(-1)

# object vertices, faces
mesh_v = so_mesh.vertices.clone()
mesh_f = so_mesh.faces.clone()

# reset variable
screenshot_plot = None

# first, create a screenshot plot with floor mesh
screenshot_plot = k3d.plot(camera_auto_fit=False)
screenshot_plot.camera = [1.5, 1, 1.5, 0, -0.25, 0, 0, 1, 0]
screenshot_plot.grid_visible = False
screenshot_plot.axes_helper = False
screenshot_plot.menu_visibility = False

# floor mesh
floormesh_plot = k3d.mesh(v.cpu().detach().numpy(), f.cpu().detach().numpy(), color=0x0000ff, shader='3d')
screenshot_plot += floormesh_plot

# create a plot for object
screenshot_z = states[0]
mesh_v_i = get_object_deformed_pts(mesh_v, screenshot_z)
mesh_v_i = mesh_v_i.squeeze(1).squeeze(1).cpu().detach().numpy()
objmesh_plot = k3d.mesh(mesh_v_i, so_mesh.faces.cpu().detach().numpy(), color=0x00ff00, shader='3d')
screenshot_plot += objmesh_plot

screenshot_plot.display()

# run in background and get screenshot
@screenshot_plot.yield_screenshots
def coroutine():
    global mesh_v_i
    for t in screenshot_timestep:
        
        # solve vertices position
        screenshot_z = states[t]
        mesh_v_i = get_object_deformed_pts(mesh_v, screenshot_z)
        mesh_v_i = mesh_v_i.squeeze(1).squeeze(1).cpu().detach().numpy()

        # objmesh_plot = k3d.mesh(mesh_v_i, so_mesh.faces.cpu().detach().numpy(), color=0x00ff00, shader='3d')
        objmesh_plot.vertices = mesh_v_i

        screenshot_plot.fetch_screenshot()

        screenshot = yield

        with open(f'./tmp/screenshot{t}.png', 'wb') as f:
            f.write(screenshot)
        with out:
            print(f'./tmp/screenshot{t}.png saved')

    with out:
        print('done')

coroutine()
print('runing in background')
out